In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
df = pd.read_csv("Churn_Modelling.csv")
print(df.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

In [3]:
X = df.drop(columns=["RowNumber", "CustomerId", "Surname", "Exited"])
y = df["Exited"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), ["Geography", "Gender"]),
], remainder="passthrough")

In [6]:
X_train = pre.fit_transform(X_train)
X_test = pre.transform(X_test)

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

In [9]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 337 (1.32 KB)

 Trainable params: 337 (1.32 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [12]:
model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early],
    verbose=2
)

Epoch 1/100
200/200 - 2s - 12ms/step - accuracy: 0.7011 - loss: 0.5759 - val_accuracy: 0.8050 - val_loss: 0.4605
Epoch 2/100
200/200 - 0s - 2ms/step - accuracy: 0.7986 - loss: 0.4638 - val_accuracy: 0.8150 - val_loss: 0.4324
Epoch 3/100
200/200 - 0s - 2ms/step - accuracy: 0.8072 - loss: 0.4418 - val_accuracy: 0.8269 - val_loss: 0.4196
Epoch 4/100
200/200 - 1s - 5ms/step - accuracy: 0.8175 - loss: 0.4289 - val_accuracy: 0.8350 - val_loss: 0.4111
Epoch 5/100
200/200 - 2s - 11ms/step - accuracy: 0.8220 - loss: 0.4234 - val_accuracy: 0.8350 - val_loss: 0.4041
Epoch 6/100
200/200 - 2s - 9ms/step - accuracy: 0.8228 - loss: 0.4140 - val_accuracy: 0.8369 - val_loss: 0.3974
Epoch 7/100
200/200 - 2s - 11ms/step - accuracy: 0.8264 - loss: 0.4077 - val_accuracy: 0.8400 - val_loss: 0.3914
Epoch 8/100
200/200 - 2s - 8ms/step - accuracy: 0.8298 - loss: 0.4037 - val_accuracy: 0.8413 - val_loss: 0.3856
Epoch 9/100
200/200 - 1s - 7ms/step - accuracy: 0.8338 - loss: 0.3992 - val_accuracy: 0.8475 - val_lo

In [13]:
prob = model.predict(X_test).ravel()

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [14]:
pred = (prob > 0.5).astype(int)

In [15]:
print("\nAccuracy:", round(accuracy_score(y_test, pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))


Accuracy: 0.864
Confusion matrix:
 [[1555   38]
 [ 234  173]]
